# Phase 4: Alpha inflation under the null
**Question:** separate "the effect is real" from "peeking is a problem regardless".

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
df = pd.read_csv('cookie_cats.csv').copy()
metric = 'retention_7'
checkpoints_frac = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

In [3]:
def run_null_simulation(seed):
    rng = np.random.default_rng(seed)
    shuffled = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    shuffled['version'] = rng.permutation(shuffled['version'].values)
    n_total = len(shuffled)
    pvals = []
    for frac in checkpoints_frac:
        n_cut = int(n_total*frac)
        sub = shuffled.iloc[:n_cut]
        g30 = sub[sub['version']=='gate_30'][metric].astype(int)
        g40 = sub[sub['version']=='gate_40'][metric].astype(int)
        if len(g30) < 30 or len(g40) < 30:
            pvals.append(np.nan); continue
        p30, p40 = g30.mean(), g40.mean()
        count = np.array([g30.sum(), g40.sum()])
        nobs = np.array([len(g30), len(g40)])
        p_pool = count.sum()/nobs.sum()
        se = np.sqrt(p_pool*(1-p_pool)*(1/nobs[0] + 1/nobs[1]))
        z = (p30-p40)/se if se>0 else 0
        p_val = 2*(1-stats.norm.cdf(abs(z)))
        pvals.append(p_val)
    return pvals

n_null_sims = 300
final_only_false_pos = 0
peeked_false_pos = 0

for seed in range(1000, 1000+n_null_sims):
    pvals = run_null_simulation(seed)
    if pvals[-1] < 0.05:
        final_only_false_pos += 1
    if any((not np.isnan(p)) and p < 0.05 for p in pvals):
        peeked_false_pos += 1

rate_final = final_only_false_pos/n_null_sims*100
rate_peek = peeked_false_pos/n_null_sims*100

print(f"Simulations under the null (no true effect): {n_null_sims}")
print(f"False-positive rate, testing ONCE at full sample only: {rate_final:.1f}%  (expected ~5%)")
print(f"False-positive rate, checking at every checkpoint and stopping on first p<0.05: {rate_peek:.1f}%")
print(f"\n-> Peeking inflated the false-positive rate by {rate_peek/rate_final:.1f}x" if rate_final>0 else "")

Simulations under the null (no true effect): 300
False-positive rate, testing ONCE at full sample only: 4.7%  (expected ~5%)
False-positive rate, checking at every checkpoint and stopping on first p<0.05: 25.0%

-> Peeking inflated the false-positive rate by 5.4x
